# Testes de Limpeza de Tags

Este notebook executa a analise de limpeza de tags para o grafo jogo-jogo do projeto.

A logica principal foi movida para `src/tag_cleaning.py` para manter a pipeline reprodutivel fora do notebook e evitar divergencia entre analise, relatorio e configuracao.

In [ ]:
from __future__ import annotations

import importlib.util
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = lambda text: text
    def display(obj):
        print(obj)

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

MODULE_PATH = BASE_DIR / "src" / "tag_cleaning.py"

spec = importlib.util.spec_from_file_location("tag_cleaning", MODULE_PATH)
tag_cleaning = importlib.util.module_from_spec(spec)
assert spec.loader is not None, "Nao foi possivel carregar src/tag_cleaning.py"
spec.loader.exec_module(tag_cleaning)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 1. Executar a pipeline

Esta chamada carrega `games_metadata.json`, aplica a configuracao em `src/config/tags.py`, executa validacoes internas e devolve todos os artefatos tabulares da analise.

In [ ]:
results = tag_cleaning.run_analysis(BASE_DIR)
stats = results["stats"]

display(pd.DataFrame([stats]).T.rename(columns={0: "valor"}))

## 2. Diagnostico e comparacao

A tabela abaixo compara o resultado antigo com a modelagem revisada.

In [ ]:
display(results["comparison_df"])
display(results["top_30_final_tags"])


## 3. Validacao semantica

Aqui verificamos explicitamente se os eixos estruturais principais foram preservados e se ainda sobraram tags nao estruturais auditadas.

In [ ]:
display(results["preserved_structural_df"])
display(results["suspicious_survivors_df"])


## 4. Impacto dos filtros

A pipeline agora depende mais de blacklist semantica e protecao de tags estruturais do que de amputacao cega por frequencia.

In [ ]:
display(results["filter_summary_df"])
display(results["too_frequent_df"])
display(results["rare_tags_df"].head(30))
